# Preprocessing Demo - placement.csv

**Steps**
1. Load and look
2. Separate numeric and categorical columns
3. Split into X and y
4. Handle missing values
5. Encode categorical columns
6. Scale numeric columns
7. Do 5 and 6 together with ColumnTransformer

**Target shape at the end: (550, 14)**

In [1]:
import pandas as pd
import numpy as np
import sklearn

---
## Step 1 - Load and look

In [2]:
df=pd.read_csv('placement.csv')
df.info()
df.head()
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 550 entries, 0 to 549
Data columns (total 12 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   student_id           550 non-null    object 
 1   cgpa                 550 non-null    float64
 2   twelfth_percent      550 non-null    float64
 3   branch               550 non-null    object 
 4   internships          519 non-null    float64
 5   projects             550 non-null    int64  
 6   backlogs             550 non-null    int64  
 7   prep_hours           454 non-null    float64
 8   communication_score  538 non-null    float64
 9   hostel               550 non-null    object 
 10  placed               550 non-null    object 
 11  package_lpa          517 non-null    float64
dtypes: float64(6), int64(2), object(4)
memory usage: 51.7+ KB


,cgpa,twelfth_percent,internships,projects,backlogs,prep_hours,communication_score,package_lpa
count,550.000000,550.000000,519.000000,550.00000,550.000000,454.000000,538.000000,517.000000
mean,7.358255,77.597455,1.302505,2.56000,0.683636,153.151982,6.269331,4.711857
std,0.983022,10.121533,1.181834,1.49164,1.097233,86.055100,1.689150,3.475017
min,5.000000,55.000000,0.000000,0.00000,0.000000,16.000000,1.100000,0.000000
25%,6.690000,70.900000,0.000000,1.00000,0.000000,90.000000,5.200000,0.000000
50%,7.400000,77.900000,1.000000,2.00000,0.000000,134.000000,6.300000,6.010000
75%,7.975000,84.500000,2.000000,4.00000,1.000000,192.750000,7.400000,7.280000
max,10.000000,98.000000,4.000000,6.00000,4.000000,500.000000,10.000000,12.560000


---
## Step 2 - Separate numeric and categorical columns

In [3]:
df.describe().columns


Index(['cgpa', 'twelfth_percent', 'internships', 'projects', 'backlogs',
       'prep_hours', 'communication_score', 'package_lpa'],
      dtype='object')

In [4]:
nums_col=['cgpa', 'twelfth_percent', 'internships', 'projects', 'backlogs',
       'prep_hours', 'communication_score']
cat_col=['branch','hostel']

---
## Step 3 - Split into X and y

In [5]:
x=df.drop(columns=['student_id','placed','package_lpa'])
y=df['placed']

---
## Step 4 - Handle missing values

In [6]:
x.isna().sum()

cgpa                    0
twelfth_percent         0
branch                  0
internships            31
projects                0
backlogs                0
prep_hours             96
communication_score    12
hostel                  0
dtype: int64

In [7]:
for col in nums_col:
    x[col]=x[col].fillna(df[col].mean())

In [8]:
df.isna().sum()

student_id              0
cgpa                    0
twelfth_percent         0
branch                  0
internships            31
projects                0
backlogs                0
prep_hours             96
communication_score    12
hostel                  0
placed                  0
package_lpa            33
dtype: int64

---
## Step 5 - Encode categorical columns

In [9]:
print(cat_col)

['branch', 'hostel']


In [10]:
df['branch'].value_counts()
df['hostel'].value_counts()

hostel
Yes    317
No     233
Name: count, dtype: int64

In [11]:
from sklearn.preprocessing import OneHotEncoder


In [12]:
encoder=OneHotEncoder()
encoded=encoder.fit_transform(x[cat_col])
print(encoded.shape)

(550, 7)


In [13]:
print(x.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 550 entries, 0 to 549
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   cgpa                 550 non-null    float64
 1   twelfth_percent      550 non-null    float64
 2   branch               550 non-null    object 
 3   internships          550 non-null    float64
 4   projects             550 non-null    int64  
 5   backlogs             550 non-null    int64  
 6   prep_hours           550 non-null    float64
 7   communication_score  550 non-null    float64
 8   hostel               550 non-null    object 
dtypes: float64(5), int64(2), object(2)
memory usage: 38.8+ KB
None


In [14]:
print(type(encoded))

<class 'scipy.sparse._csr.csr_matrix'>


---
## Step 6 - Scale numeric columns

In [15]:
from sklearn.preprocessing import StandardScaler
scaler=StandardScaler()
scaled_col=scaler.fit_transform(x[nums_col])
scaled_col.shape


(550, 7)

---
## Step 7 - Both together with ColumnTransformer

In [16]:
from sklearn.compose import ColumnTransformer


In [17]:
preprocessor=ColumnTransformer([
    ('cat',OneHotEncoder(),cat_col),
    ('num',StandardScaler(),nums_col)
],remainder='passthrough')



In [18]:
final_transform = preprocessor.fit_transform(x)

In [20]:
final_transform.shape

(550, 14)

In [21]:
preprocessor.get_feature_names_out()

array(['cat__branch_CSE', 'cat__branch_Civil', 'cat__branch_ECE',
       'cat__branch_IT', 'cat__branch_Mech', 'cat__hostel_No',
       'cat__hostel_Yes', 'num__cgpa', 'num__twelfth_percent',
       'num__internships', 'num__projects', 'num__backlogs',
       'num__prep_hours', 'num__communication_score'], dtype=object)